In [88]:
import os
import glob
import mne
import pandas as pd
import numpy as np

# ==========================================
# 1. تحديد مسارات المجلد وتسمية ملف الخرج
# ==========================================
folder_path =   r"C:\Users\USER\Downloads\archive (5)\files\S001" 
output_csv_path =r"C:\Users\USER\Downloads\eeg_hands_dataset.csv"

# أرقام التجارب الخاصة باليدين فقط في PhysioNet dataset
hand_runs = ['R03', 'R04', 'R07', 'R08', 'R11', 'R12']

# جلب وتصفية الملفات التي تنتهي بأرقام تجارب اليدين فقط (مع استبعاد ملفات .edf.event)
all_edf_files = sorted(glob.glob(os.path.join(folder_path, "S*R*.edf")))
hand_edf_files = [f for f in all_edf_files if any(run in os.path.basename(f) for run in hand_runs)]

all_epochs_dfs = []
global_trial_id = 1  # معرّف موحد وفريد لكل تجربة (trial_id) عبر كل الملفات

print(f"تم العثور على {len(hand_edf_files)} ملفاً خاصاً بتجارب اليدين.")

# ==========================================
# 2. المرور على ملفات اليدين وتطبيق الـ Epoching
# ==========================================
for file_path in hand_edf_files:
    file_name = os.path.basename(file_path)
    subject_id = file_name.split('R')[0]  # استخراج اسم المشارك (مثل S001)
    run_id = 'R' + file_name.split('R')[1].replace('.edf', '')  # استخراج رقم الـ Run
    
    # قراءة بيانات الـ EDF
    raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)
    
    # تحضير الأحداث وتصفية Left Hand (T1) و Right Hand (T2)
    events, event_id = mne.events_from_annotations(raw, verbose=False)
    
    # ربط T1 بـ Left Hand و T2 بـ Right Hand
    custom_event_id = {}
    if 'T1' in event_id: custom_event_id['Left Hand'] = event_id['T1']
    if 'T2' in event_id: custom_event_id['Right Hand'] = event_id['T2']
    
    if not custom_event_id:
        continue

    # تقطيع الإشارات (Epoching) لليدين فقط (بدون T0 / Rest)
    epochs = mne.Epochs(
        raw, 
        events=events, 
        event_id=custom_event_id, 
        tmin=0.0, 
        tmax=4.0, 
        baseline=None, 
        preload=True,
        verbose=False
    )
    
    # تحويل الـ Epochs الحالية إلى DataFrame
    df_single_file = epochs.to_data_frame()
    
    # ==========================================
    # 3. إعداد الـ Metadata وترقيم الـ trial_id
    # ==========================================
    # تحويل الترقيم الداخلي للـ epoch ليكون فريداً وموحداً (global trial_id)
    local_epochs = df_single_file['epoch'].unique()
    epoch_to_trial_map = {loc_ep: global_trial_id + i for i, loc_ep in enumerate(local_epochs)}
    
    df_single_file['trial_id'] = df_single_file['epoch'].map(epoch_to_trial_map)
    global_trial_id += len(local_epochs)  # تحديث العداد للملف القادم
    
    # إضافة الميتاداتا الوصفية للملف
    df_single_file['subject'] = subject_id
    df_single_file['run'] = run_id
    df_single_file['label'] = df_single_file['condition']  # Left Hand / Right Hand
    
    # حذف الأعمدة التكرارية أو القديمة لتنظيم الجدول
    df_single_file.drop(columns=['condition', 'epoch'], inplace=True)
    
    all_epochs_dfs.append(df_single_file)
    print(f"تمت معالجة {file_name} | التجميع الحالي للأحداث: {len(local_epochs)} تجربة.")

# ==========================================
# 4. دمج البيانات وحفظ الـ CSV الموحد
# ==========================================
if all_epochs_dfs:
    final_df = pd.concat(all_epochs_dfs, ignore_index=True)
    
    # ترتيب الأعمدة لوضع الـ Metadata في البداية ثم قنوات الـ EEG
    metadata_cols = ['trial_id', 'subject', 'run', 'label', 'time']
    eeg_cols = [col for col in final_df.columns if col not in metadata_cols]
    final_df = final_df[metadata_cols + eeg_cols]
    
    # حفظ الـ DataFrame نهائياً في ملف CSV
    final_df.to_csv(output_csv_path, index=False)
    print(f"\n تم حفظ الـ Dataset بنجاح في: {output_csv_path}")
    print(f"إجمالي عدد الصفوف: {len(final_df)} | إجمالي عدد التجربات (trials): {final_df['trial_id'].nunique()}")
else:
    print("لم يتم العثور على أي ملفات مطابقة لشروط اليدين.")

تم العثور على 6 ملفاً خاصاً بتجارب اليدين.
تمت معالجة S001R03.edf | التجميع الحالي للأحداث: 15 تجربة.
تمت معالجة S001R04.edf | التجميع الحالي للأحداث: 15 تجربة.
تمت معالجة S001R07.edf | التجميع الحالي للأحداث: 15 تجربة.
تمت معالجة S001R08.edf | التجميع الحالي للأحداث: 15 تجربة.
تمت معالجة S001R11.edf | التجميع الحالي للأحداث: 15 تجربة.
تمت معالجة S001R12.edf | التجميع الحالي للأحداث: 15 تجربة.

 تم حفظ الـ Dataset بنجاح في: C:\Users\USER\Downloads\eeg_hands_dataset.csv
إجمالي عدد الصفوف: 57690 | إجمالي عدد التجربات (trials): 90


In [99]:
e = pd.read_csv(r"C:\Users\USER\Downloads\eeg_hands_dataset.csv")
roi_channels = ['trial_id', 'subject', 'run', 'label', 'time',
    'Fc5.', 'Fc3.', 'Fc1.', 'Fcz.', 'Fc2.', 'Fc4.', 'Fc6.',
    'C5..',  'C3..',  'C1..',  'Cz..',  'C2..',  'C4..',  'C6..',
    'Cp5.', 'Cp3.', 'Cp1.', 'Cpz.', 'Cp2.', 'Cp4.', 'Cp6.'
]
e = e[roi_channels]

In [100]:
e.columns

Index(['trial_id', 'subject', 'run', 'label', 'time', 'Fc5.', 'Fc3.', 'Fc1.',
       'Fcz.', 'Fc2.', 'Fc4.', 'Fc6.', 'C5..', 'C3..', 'C1..', 'Cz..', 'C2..',
       'C4..', 'C6..', 'Cp5.', 'Cp3.', 'Cp1.', 'Cpz.', 'Cp2.', 'Cp4.', 'Cp6.'],
      dtype='object')

In [101]:
e

,trial_id,subject,run,label,time,Fc5.,Fc3.,Fc1.,Fcz.,Fc2.,...,C2..,C4..,C6..,Cp5.,Cp3.,Cp1.,Cpz.,Cp2.,Cp4.,Cp6.
0,1,S001,R03,Right Hand,0.00000,-24.0,-8.0,-4.0,-26.0,-22.0,...,19.0,-12.0,-12.0,18.0,20.0,14.0,11.0,8.0,-8.0,-20.0
1,1,S001,R03,Right Hand,0.00625,-28.0,-11.0,-12.0,-35.0,-27.0,...,16.0,-18.0,-7.0,0.0,9.0,4.0,-3.0,-2.0,-16.0,-22.0
2,1,S001,R03,Right Hand,0.01250,-33.0,-29.0,-37.0,-65.0,-58.0,...,-15.0,-52.0,-31.0,-20.0,-20.0,-30.0,-41.0,-34.0,-48.0,-45.0
3,1,S001,R03,Right Hand,0.01875,-36.0,-22.0,-23.0,-51.0,-41.0,...,3.0,-35.0,-14.0,-20.0,-20.0,-24.0,-26.0,-17.0,-32.0,-27.0
4,1,S001,R03,Right Hand,0.02500,-16.0,12.0,16.0,-12.0,-3.0,...,23.0,-17.0,-8.0,-13.0,-6.0,-9.0,-9.0,-6.0,-22.0,-25.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57685,90,S001,R12,Right Hand,3.97500,-1.0,11.0,20.0,21.0,25.0,...,14.0,4.0,5.0,-26.0,-13.0,-8.0,-5.0,-7.0,-8.0,0.0
57686,90,S001,R12,Right Hand,3.98125,12.0,20.0,21.0,20.0,19.0,...,0.0,-11.0,-13.0,-26.0,-14.0,-14.0,-13.0,-23.0,-25.0,-17.0
57687,90,S001,R12,Right Hand,3.98750,10.0,23.0,26.0,21.0,20.0,...,-8.0,-15.0,-12.0,-33.0,-23.0,-22.0,-17.0,-29.0,-31.0,-20.0
57688,90,S001,R12,Right Hand,3.99375,11.0,26.0,24.0,16.0,12.0,...,-12.0,-25.0,-15.0,-41.0,-35.0,-30.0,-23.0,-35.0,-35.0,-25.0


In [102]:
e.to_csv(r"C:\Users\USER\Downloads\eeg_hands_df.csv")